In [1]:
# ==============================
# Federated Learning on Fashion-MNIST (Dirichlet Non-IID)
# Model: LeNet++ CNN
# Algorithm: FedSGD ONLY
# Tracks: Train/Val/Test Loss & Accuracy + Round Time + Total Time
# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import random
import copy
import time
from collections import OrderedDict

# -------------------------
# Configuration
# -------------------------
NUM_CLIENTS = 100
CLIENT_FRAC = 0.3
ROUNDS = 60
LOCAL_EPOCHS = 1
BATCH_SIZE = 128
LR = 0.01
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
ALPHA = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Device:", DEVICE)

# -------------------------
# Dataset: Fashion-MNIST
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# -------------------------
# Clients split
# -------------------------
VAL_CLIENTS = list(range(10))
TRAIN_CLIENTS = list(range(10, NUM_CLIENTS))

# -------------------------
# Dirichlet Non-IID
# -------------------------
def dirichlet_partition(dataset, clients, alpha=0.1):

    num_classes = 10

    labels = np.array(dataset.targets)

    idxs = np.arange(len(labels))

    client_dict = {i: [] for i in clients}

    class_indices = [
        idxs[labels == i]
        for i in range(num_classes)
    ]

    for c_idx in class_indices:

        np.random.shuffle(c_idx)

        proportions = np.random.dirichlet(
            alpha * np.ones(len(clients))
        )

        proportions = (
            np.cumsum(proportions) * len(c_idx)
        ).astype(int)

        proportions = np.concatenate(([0], proportions))

        for i, client in enumerate(clients):

            client_dict[client] += list(
                c_idx[proportions[i]:proportions[i+1]]
            )

    for client in clients:
        np.random.shuffle(client_dict[client])

    return client_dict

client_data = dirichlet_partition(
    train_dataset,
    list(range(NUM_CLIENTS)),
    ALPHA
)

# -------------------------
# Model
# -------------------------
class FMNIST_CNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = nn.Conv2d(
            1,
            32,
            3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            32,
            64,
            3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(
            64 * 7 * 7,
            256
        )

        self.fc2 = nn.Linear(
            256,
            10
        )

    def forward(self, x):

        x = self.pool(
            F.relu(self.conv1(x))
        )

        x = self.pool(
            F.relu(self.conv2(x))
        )

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))

        return self.fc2(x)

# -------------------------
# Evaluation
# -------------------------
def evaluate(model, loader):

    model.eval()

    correct, total, loss_val = 0, 0, 0.0

    with torch.no_grad():

        for x, y in loader:

            x, y = x.to(DEVICE), y.to(DEVICE)

            out = model(x)

            loss_val += (
                F.cross_entropy(out, y).item()
                * y.size(0)
            )

            pred = out.argmax(dim=1)

            correct += (pred == y).sum().item()

            total += y.size(0)

    return (
        100 * correct / total,
        loss_val / total
    )

# -------------------------
# Federated Training (FedSGD ONLY)
# -------------------------
global_model = FMNIST_CNN().to(DEVICE)

train_acc, val_acc, test_acc = [], [], []

loss_curve = []
train_loss_curve = []
val_loss_curve = []
test_loss_curve = []

round_time, total_time = [], []

start_total = time.time()

for rnd in range(ROUNDS):

    start_round = time.time()

    selected_clients = random.sample(
        TRAIN_CLIENTS,
        int(NUM_CLIENTS * CLIENT_FRAC)
    )

    local_gradients = []
    local_sizes = []
    local_losses = []

    # -------------------------
    # Local gradient computation
    # -------------------------
    for cid in selected_clients:

        local_model = copy.deepcopy(global_model)

        local_model.train()

        loader = DataLoader(
            Subset(train_dataset, client_data[cid]),
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        gradient_sums = OrderedDict()

        for name, param in local_model.named_parameters():
            gradient_sums[name] = torch.zeros_like(param.data)

        batch_count = 0

        for x, y in loader:

            x, y = x.to(DEVICE), y.to(DEVICE)

            local_model.zero_grad()

            out = local_model(x)

            loss = F.cross_entropy(out, y)

            loss.backward()

            for name, param in local_model.named_parameters():
                gradient_sums[name] += param.grad.data.clone()

            batch_count += 1

        for name in gradient_sums:
            gradient_sums[name] /= batch_count

        local_gradients.append(copy.deepcopy(gradient_sums))

        local_sizes.append(len(client_data[cid]))

        local_losses.append(loss.item())

    # -------------------------
    # FedSGD aggregation
    # -------------------------
    total = sum(local_sizes)

    global_weights = global_model.state_dict()

    for name, param in global_model.named_parameters():

        avg_grad = sum(
            local_gradients[i][name]
            * (local_sizes[i] / total)
            for i in range(len(local_gradients))
        )

        global_weights[name] = (
            global_weights[name]
            - LR * avg_grad
        )

    global_model.load_state_dict(global_weights)

    # -------------------------
    # Metrics
    # -------------------------
    tr_acc, tr_loss = evaluate(
        global_model,
        DataLoader(
            torch.utils.data.ConcatDataset([
                Subset(train_dataset, client_data[c])
                for c in selected_clients
            ]),
            batch_size=BATCH_SIZE
        )
    )

    va_acc, va_loss = evaluate(
        global_model,
        DataLoader(
            torch.utils.data.ConcatDataset([
                Subset(train_dataset, client_data[c])
                for c in VAL_CLIENTS
            ]),
            batch_size=BATCH_SIZE
        )
    )

    te_acc, te_loss = evaluate(
        global_model,
        DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE
        )
    )

    # logs
    loss_curve.append(np.mean(local_losses))

    train_loss_curve.append(tr_loss)
    val_loss_curve.append(va_loss)
    test_loss_curve.append(te_loss)

    train_acc.append(tr_acc)
    val_acc.append(va_acc)
    test_acc.append(te_acc)

    rt = time.time() - start_round

    round_time.append(rt)

    total_time.append(
        time.time() - start_total
    )

    print(
        f"Round {rnd+1:03d} | "
        f"LocalLoss {loss_curve[-1]:.4f} | "
        f"TrainLoss {tr_loss:.4f} | "
        f"ValLoss {va_loss:.4f} | "
        f"TestLoss {te_loss:.4f} | "
        f"Train {tr_acc:.4f} | "
        f"Val {va_acc:.4f} | "
        f"Test {te_acc:.4f} | "
        f"RoundTime {rt:.2f}s | "
        f"TotalTime {total_time[-1]:.2f}s"
    )

Device: cuda


100%|██████████| 26.4M/26.4M [00:02<00:00, 12.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 199kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.79MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.3MB/s]


Round 001 | LocalLoss 2.2948 | TrainLoss 2.2833 | ValLoss 2.3125 | TestLoss 2.3013 | Train 14.5509 | Val 17.7197 | Test 10.9600 | RoundTime 13.86s | TotalTime 13.86s
Round 002 | LocalLoss 2.3076 | TrainLoss 2.2963 | ValLoss 2.3105 | TestLoss 2.2989 | Train 12.7386 | Val 17.4534 | Test 11.7000 | RoundTime 11.01s | TotalTime 24.86s
Round 003 | LocalLoss 2.2904 | TrainLoss 2.2854 | ValLoss 2.3102 | TestLoss 2.2965 | Train 11.6859 | Val 14.5243 | Test 12.8600 | RoundTime 12.36s | TotalTime 37.22s
Round 004 | LocalLoss 2.2955 | TrainLoss 2.2867 | ValLoss 2.3043 | TestLoss 2.2936 | Train 14.2234 | Val 17.9860 | Test 12.2200 | RoundTime 9.39s | TotalTime 46.61s
Round 005 | LocalLoss 2.2916 | TrainLoss 2.2766 | ValLoss 2.3049 | TestLoss 2.2907 | Train 18.9232 | Val 18.0586 | Test 14.6400 | RoundTime 11.26s | TotalTime 57.87s
Round 006 | LocalLoss 2.2837 | TrainLoss 2.2825 | ValLoss 2.3050 | TestLoss 2.2887 | Train 16.3377 | Val 17.6955 | Test 16.6500 | RoundTime 13.58s | TotalTime 71.45s
Round

In [ ]:
# -------------------------
# Save metrics to CSV
# -------------------------
import csv

metrics = {
    "round": list(range(1, ROUNDS+1)),
    "train_acc": train_acc,
    "val_acc": val_acc,
    "test_acc": test_acc,
    "local_loss": loss_curve,
    "train_loss": train_loss_curve,
    "val_loss": val_loss_curve,
    "test_loss": test_loss_curve,
    "round_time": round_time,
    "total_time": total_time
}

with open("FMNIST_fedsgd_federated_metrics_0.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(metrics.keys())
    writer.writerows(zip(*metrics.values()))

# Colab download
try:
    from google.colab import files
    files.download("FMNIST_fedsgd_federated_metrics_0.csv")
except:
    print("CSV saved locally as FMNIST_fedsgd_federated_metrics_0.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>